# ETL Mini Case Study — Lists, Tuples, and Dictionaries Only
_Generated: 2026-03-02 08:41:21_

This notebook implements a small, self-contained **ETL pipeline** using only Python core data structures: **lists, tuples, and dictionaries**. It follows a phased approach and keeps the logic explicit and readable.

## Problem Statement

We receive raw vehicle telemetry as tuples in the shape `(vehicle_id, timestamp_iso, speed_kmh, battery_temp_c, status_code)`.

**Data quality challenges to solve:**

1. **Duplicates**: Multiple rows may repeat the same `(vehicle_id, timestamp_iso)` combination.
2. **Missing values**: `battery_temp_c` may be `None`.
3. **Invalid ranges**: `speed_kmh` outside `[0, 250]` or `battery_temp_c` outside `[-50, 120]` must be treated as invalid.
4. **Opaque codes**: `status_code` (0/1/2) must be mapped to human-readable labels (`OK`, `WARN`, `FAULT`).

**Outputs required:**

- Clean, de-duplicated **records** as dictionaries (names over positions).
- Per-vehicle **KPIs** based on cleaned data:
  - `max_speed_kmh`
  - `avg_battery_temp_c` (mean of valid temps only)
  - `fault_count` (count of rows with status == `FAULT`)

**Constraints:** Use only lists, tuples, and dictionaries for the pipeline logic (no external libraries).

**Non-functional goals:** Code should be **readable**, **traceable**, and **robust** to imperfect input.


## Key Objectives (3)

1. **Semantic Modeling**: Convert tuples to dictionaries early, so fields are accessed by name rather than position.
2. **Data Quality**: Deduplicate, validate ranges, and handle missing/invalid values deterministically.
3. **Actionable KPIs**: Aggregate per-vehicle metrics (max, average, counts) suitable for downstream use.


_Hands-on note: A Python Jupyter notebook is provided for practice—download it and execute the cells locally to follow along step by step._

## P0 — Rules & Schema
Define domain rules and helper mappings used throughout the pipeline.

In [ ]:
# Column order in each input tuple:
# (vehicle_id: str, timestamp_iso: str, speed_kmh: int, battery_temp_c: int|None, status_code: int)

SPEED_MIN, SPEED_MAX = 0, 250
TEMP_MIN,  TEMP_MAX  = -50, 120

STATUS_MAP = {
    0: 'OK',
    1: 'WARN',
    2: 'FAULT',
}


## P1 — Extract (Raw Tuples)
Simulate a raw extract as a list of tuples with duplicates and invalid data present.

In [ ]:
raw_rows = [
    ('V1', '2026-03-01T10:00:00Z',  60,  35, 0),
    ('V1', '2026-03-01T10:01:00Z',  62,  36, 0),
    ('V2', '2026-03-01T10:00:00Z', 200,  80, 1),
    ('V1', '2026-03-01T10:01:00Z',  62,  None, 0),  # duplicate timestamp for V1
    ('V3', '2026-03-01T10:00:00Z', -10,  30, 0),    # invalid speed (negative)
    ('V2', '2026-03-01T10:02:00Z', 260,  90, 2),    # invalid speed (too high)
    ('V2', '2026-03-01T10:03:00Z', 120, 180, 2),    # invalid temp (too high)
]
len(raw_rows)


## P2 — Normalize (Tuples → Dictionaries)
Convert tuples into dictionaries for semantic access (names over positions).

In [ ]:
def tuple_to_dict(row):
    vehicle_id, ts, speed, temp, code = row
    return {
        'vehicle_id': vehicle_id,
        'timestamp': ts,
        'speed_kmh': speed,
        'battery_temp_c': temp,
        'status_code': code,
    }

records = [tuple_to_dict(r) for r in raw_rows]
records[:3]  # preview

## P3 — Enrich (Map Status Codes to Labels)
Add human-readable labels (OK/WARN/FAULT) derived from `status_code`.

In [ ]:
for r in records:
    r['status_label'] = STATUS_MAP.get(r['status_code'], 'UNKNOWN')
records[:3]  # preview

## P4 — De-duplicate (by Composite Key)
Keep the first occurrence of each `(vehicle_id, timestamp)` pair.

In [ ]:
seen = {}
deduped = []
for r in records:
    key = (r['vehicle_id'], r['timestamp'])
    if key not in seen:
        seen[key] = True
        deduped.append(r)

len(records), len(deduped)


## P5 — Validate & Clean
- Drop rows with invalid speed.
- Clamp invalid temperatures to `None` (retain row).

In [ ]:
cleaned = []
for r in deduped:
    speed = r['speed_kmh']
    temp  = r['battery_temp_c']

    # Speed validation (row-level)
    if not (SPEED_MIN <= speed <= SPEED_MAX):
        continue  # drop row

    # Temperature validation (field-level)
    if temp is not None and not (TEMP_MIN <= temp <= TEMP_MAX):
        r = dict(r)  # shallow copy before mutation
        r['battery_temp_c'] = None

    cleaned.append(r)

len(deduped), len(cleaned)


## P6 — Aggregate KPIs (Per Vehicle)
Compute `max_speed_kmh`, `avg_battery_temp_c` (over non-None temps), and `fault_count`.

In [ ]:
agg = {}  # vehicle_id -> accumulators

for r in cleaned:
    vid = r['vehicle_id']
    if vid not in agg:
        agg[vid] = {
            'max_speed_kmh': None,
            'temp_sum': 0,
            'temp_count': 0,
            'fault_count': 0,
        }

    a = agg[vid]
    s = r['speed_kmh']
    if a['max_speed_kmh'] is None or s > a['max_speed_kmh']:
        a['max_speed_kmh'] = s

    t = r['battery_temp_c']
    if t is not None:
        a['temp_sum'] += t
        a['temp_count'] += 1

    if r['status_label'] == 'FAULT':
        a['fault_count'] += 1

# finalize avg
for vid, a in agg.items():
    if a['temp_count'] > 0:
        avg_temp = a['temp_sum'] / a['temp_count']
    else:
        avg_temp = None
    a['avg_battery_temp_c'] = avg_temp
    del a['temp_sum'], a['temp_count']

agg


## P7 — Load (Compose Final Structure)
Assemble a final dictionary keyed by `vehicle_id`, embedding KPIs and cleaned events.

In [ ]:
# Index cleaned events by vehicle
by_vehicle = {}
for r in cleaned:
    vid = r['vehicle_id']
    by_vehicle.setdefault(vid, []).append(r)

# Compose final payload
final = {}
for vid, events in by_vehicle.items():
    final[vid] = {
        'kpis': agg.get(vid, {
            'max_speed_kmh': None,
            'avg_battery_temp_c': None,
            'fault_count': 0,
        }),
        'events': events,
    }

list(final.keys())


## P8 — QA Checks
Lightweight integrity checks to keep results trustworthy.

In [ ]:
# (1) Every cleaned event is present in 'final'
loaded_event_count = sum(len(v['events']) for v in final.values())
assert loaded_event_count == len(cleaned), 'Mismatch between cleaned and loaded events'

# (2) KPI keys exist for each vehicle
for vid, payload in final.items():
    for k in ('max_speed_kmh', 'avg_battery_temp_c', 'fault_count'):
        assert k in payload['kpis'], f'Missing KPI {k} for vehicle {vid}'
'QA checks passed.'


## Final Payload (Pretty Print)
Drill into results conveniently; below we pretty-print the entire structure.

In [ ]:
from pprint import pprint
pprint(final)
